# Распознавание овощей — Часть 1 (разбор данных) + Трек B (классические модели)

**Как запускать:** Colab → Runtime → Change runtime type → **T4 GPU** → Runtime → Run all.
GPU нужен только чтобы быстро посчитать эмбеддинги (~2–3 мин). Всё остальное — sklearn, секунды.

В конце ноутбук сам соберёт папку `repo/` (predict.py, веса, requirements.txt, графики) и скачает `repo.zip`.

In [1]:
!pip -q install kagglehub imagehash


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os, io, re, json, hashlib, random, shutil
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image, ImageOps
import imagehash
from skimage.feature import hog
from joblib import Parallel, delayed
from tqdm.auto import tqdm
import torch, torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, log_loss)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
CLASSES = ['Capsicum', 'Carrot', 'Cucumber', 'Potato', 'Raddish', 'Squash']
EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
REPO = Path('/content/repo'); (REPO / 'weights').mkdir(parents=True, exist_ok=True)
FIG = REPO / 'figures'; FIG.mkdir(exist_ok=True)
RESULTS = {}
print('device:', DEVICE)

ModuleNotFoundError: No module named 'matplotlib'

## 0. Скачиваем датасет и смотрим структуру папок

In [ ]:
import kagglehub
DATA = Path(kagglehub.dataset_download('preethi866/vegetable-dataset'))
print(DATA)
for root, dirs, files in os.walk(DATA):
    depth = len(Path(root).relative_to(DATA).parts)
    if depth <= 3:
        print('  ' * depth + Path(root).name + f'/  ({len(files)} files)')

In [ ]:
def norm(name):
    # 'Raddish' / 'Radish' -> same key
    return name.lower().replace('dd', 'd')

def class_dirs(root):
    # Return {class: folder} if root contains all 6 class folders, else None
    sub = {norm(d.name): d for d in Path(root).iterdir() if d.is_dir()}
    found = {c: sub.get(norm(c)) for c in CLASSES}
    return found if all(found.values()) else None

roots = [Path(r) for r, _, _ in os.walk(DATA) if class_dirs(r)]
rel = lambda r: str(r.relative_to(DATA)).lower()
for r in roots: print(r.relative_to(DATA))

# If auto-detection picks a wrong folder, set these 3 paths by hand
ALL_ROOT   = next(r for r in roots if 'final_dataset' not in rel(r))
TRAIN_ROOT = next(r for r in roots if 'final_dataset' in rel(r) and 'train' in rel(r))
TEST_ROOT  = next(r for r in roots if 'final_dataset' in rel(r) and 'test' in rel(r))
print('ALL  :', ALL_ROOT); print('TRAIN:', TRAIN_ROOT); print('TEST :', TEST_ROOT)

In [ ]:
def list_images(root):
    rows = []
    for c, d in class_dirs(root).items():
        for p in sorted(d.rglob('*')):
            if p.is_file():
                rows.append({'path': str(p), 'label': c, 'name': p.name, 'ext': p.suffix.lower()})
    t = pd.DataFrame(rows)
    print(root.name, '| all file types:', t.ext.value_counts().to_dict())
    return t[t.ext.isin(EXTS)].reset_index(drop=True)

df    = list_images(ALL_ROOT)
tr_df = list_images(TRAIN_ROOT)
te_df = list_images(TEST_ROOT)
print(len(df), 'images in Dataset/ |', len(tr_df), 'train +', len(te_df), 'test in final_dataset/')

# Часть 1. Разбор данных

### 1.1 Читаем каждое фото один раз: размер, формат, хеши и простые признаки

- **md5** — хеш байтов файла. Одинаковый md5 = файл скопирован 1-в-1.
- **perceptual hash (pHash)** — «отпечаток» того, *как картинка выглядит*. Если фото пережали,
  чуть изменили яркость или размер — pHash почти не меняется. Расстояние между хешами
  (сколько из 64 битов отличаются) = насколько фото похожи.
- **простые признаки** для Трека B: цветовая гистограмма HSV + HOG (форма/контуры). Считаем сразу, чтобы не читать фото дважды.

In [ ]:
def load_rgb(path, max_side=320):
    # Same loader is used in predict.py
    img = Image.open(path)
    img.draft('RGB', (max_side, max_side))   # fast JPEG downscale while decoding
    img = ImageOps.exif_transpose(img)       # phone photos: respect EXIF rotation
    return img.convert('RGB')

def simple_features(img):
    small = img.resize((128, 128))
    hsv = np.asarray(small.convert('HSV'))
    hist = np.concatenate([np.histogram(hsv[..., c], bins=16, range=(0, 256))[0] for c in range(3)]).astype(np.float32)
    hist /= hist.sum()
    gray = np.asarray(small.convert('L'), dtype=np.float32) / 255
    h = hog(gray, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2), feature_vector=True)
    return np.concatenate([hist, h]).astype(np.float32)

def describe(path):
    rec = {}
    try:
        raw = Path(path).read_bytes()
        rec['md5'], rec['bytes'] = hashlib.md5(raw).hexdigest(), len(raw)
        im = Image.open(io.BytesIO(raw))
        rec['format'], rec['mode'] = im.format, im.mode
        rec['width'], rec['height'] = im.size
        img = load_rgb(io.BytesIO(raw))
        rec['phash'] = str(imagehash.phash(img))
        rec['feat'] = simple_features(img)
        rec['ok'] = True
    except Exception as e:
        rec['ok'], rec['error'] = False, repr(e)
    return rec

recs = Parallel(n_jobs=-1, batch_size=32)(delayed(describe)(p) for p in tqdm(df.path))
df = pd.concat([df, pd.DataFrame(recs)], axis=1)
print('broken files:', (~df.ok).sum())
if (~df.ok).any(): display(df.loc[~df.ok, ['path', 'error']].head())
df = df[df.ok].reset_index(drop=True)
X_simple = np.stack(df.pop('feat').values)
print('simple features:', X_simple.shape)

### 1.2 Сколько фото в классах, какие размеры, что говорят имена файлов

In [ ]:
counts = df.label.value_counts().reindex(CLASSES)
print(counts.to_string()); print('max/min ratio:', round(counts.max() / counts.min(), 2))
counts.plot.bar(title='Images per class'); plt.tight_layout(); plt.savefig(FIG / 'class_counts.png', dpi=120); plt.show()

print('\nformats:', df.format.value_counts().to_dict(), '| modes:', df['mode'].value_counts().to_dict())
df['size'] = df.width.astype(str) + 'x' + df.height.astype(str)
print('\nmost common sizes per class:')
for c in CLASSES:
    print(f'  {c:9s}', df[df.label == c]['size'].value_counts().head(4).to_dict())
display(df.groupby('label')[['width', 'height', 'bytes']].describe().round(0))

# File names: replace digits with '#', look at the most common patterns
df['pattern'] = df.name.map(lambda n: re.sub(r'\d+', '#', Path(n).stem))
print('\nfile-name patterns per class:')
for c in CLASSES:
    print(f'  {c:9s}', df[df.label == c].pattern.value_counts().head(6).to_dict())

In [ ]:
def show_grid(paths, titles=None, ncols=6, size=2.2, save=None):
    n = len(paths); nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * size, nrows * size))
    for ax in np.ravel(axes): ax.axis('off')
    for i, p in enumerate(paths):
        ax = np.ravel(axes)[i]; ax.imshow(load_rgb(p))
        if titles is not None: ax.set_title(titles[i], fontsize=8)
    plt.tight_layout()
    if save: plt.savefig(FIG / save, dpi=110)
    plt.show()

sample = df.groupby('label').sample(6, random_state=SEED)
show_grid(sample.path.tolist(), (sample.label + '\n' + sample.name.str[:18]).tolist(), save='samples.png')

### 1.3 Эмбеддинги замороженной ResNet18

**Эмбеддинг** — это 512 чисел, которыми предобученная сеть «описывает» фото.
Похожие фото → похожие векторы. Они нужны дважды: искать почти-дубликаты (обрезка, поворот —
то, что pHash пропускает) и как признаки для Трека B.

In [ ]:
backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
backbone.fc = torch.nn.Identity()          # drop the ImageNet head -> 512-d embedding
backbone = backbone.eval().to(DEVICE)
TF = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

class ImgDS(Dataset):
    def __init__(self, paths): self.paths = list(paths)
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return TF(load_rgb(self.paths[i]))

@torch.inference_mode()
def embed(paths, bs=64):
    dl = DataLoader(ImgDS(paths), batch_size=bs, num_workers=2)
    return np.concatenate([backbone(x.to(DEVICE)).float().cpu().numpy() for x in tqdm(dl)])

X_emb = embed(df.path)
print('embeddings:', X_emb.shape)

### 1.4 Дубликаты и почти-дубликаты

Три проверки:
1. **точные копии** — одинаковый md5;
2. **почти-копии по pHash** — расстояние Хэмминга ≤ `HAM_THR`;
3. **почти-копии по эмбеддингам** — косинусное сходство ≥ `COS_THR` (ловит кропы/повороты).

Пороги не «из головы»: смотрим гистограмму и **глазами проверяем пары** на границе.

In [ ]:
def hash_bits(hashes):
    return np.array([[int(b) for b in bin(int(h, 16))[2:].zfill(64)] for h in hashes], dtype=np.float32)

def close_pairs_hamming(A, B=None, thr=6, chunk=1000):
    # Pairs (i, j, distance) with Hamming distance <= thr. If B is None: pairs inside A, i < j.
    same = B is None
    B = A if same else B
    out = []
    for s in range(0, len(A), chunk):
        a = A[s:s + chunk]
        d = a @ (1 - B).T + (1 - a) @ B.T          # number of differing bits
        i, j = np.nonzero(d <= thr)
        dist = d[i, j]; i = i + s
        if same:
            keep = j > i; i, j, dist = i[keep], j[keep], dist[keep]
        out.append(np.stack([i, j, dist], 1))
    return np.concatenate(out) if out else np.empty((0, 3))

def cosine_scan(E, thr=0.95, chunk=1000):
    # Returns pairs (i, j, sim) with sim >= thr (i < j) and nearest-neighbour similarity per image
    E = E - E.mean(0)                                   # centre: ReLU features are all positive
    E = E / np.linalg.norm(E, axis=1, keepdims=True)
    out, nn = [], np.empty(len(E), dtype=np.float32)
    for s in range(0, len(E), chunk):
        S = E[s:s + chunk] @ E.T
        S[np.arange(len(S)), np.arange(s, s + len(S))] = -1     # ignore self-similarity
        nn[s:s + len(S)] = S.max(1)
        i, j = np.nonzero(S >= thr)
        sim = S[i, j]; i = i + s
        keep = j > i
        out.append(np.stack([i[keep], j[keep], sim[keep]], 1))
    return np.concatenate(out), nn

# 1) exact copies
dup_md5 = df[df.md5.duplicated(keep=False)]
print('exact copies (md5): files involved =', len(dup_md5), '| unique contents =', dup_md5.md5.nunique())
conflict = dup_md5.groupby('md5').label.nunique()
print('identical file under DIFFERENT classes:', (conflict > 1).sum())

# 2) and 3) distributions to choose thresholds
H = hash_bits(df.phash)
_, nn_sim = cosine_scan(X_emb, thr=1.01)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(nn_sim, bins=100); ax[0].set_title('Embedding: cosine to nearest other image'); ax[0].set_yscale('log')
pairs_any = close_pairs_hamming(H, thr=20)
ax[1].hist(pairs_any[:, 2], bins=21); ax[1].set_title('pHash: Hamming distance of close pairs (<=20)'); ax[1].set_yscale('log')
plt.tight_layout(); plt.savefig(FIG / 'dup_distributions.png', dpi=120); plt.show()

In [ ]:
# Look at pairs at different similarity levels and decide where "same photo" ends
def show_pairs(pairs, n=4, title=''):
    if len(pairs) == 0: print(title, '-> no pairs'); return
    pick = pairs[np.random.default_rng(SEED).choice(len(pairs), min(n, len(pairs)), replace=False)]
    paths, titles = [], []
    for i, j, v in pick:
        i, j = int(i), int(j)
        paths += [df.path[i], df.path[j]]
        titles += [f'{df.label[i]} {v:.3f}', f'{df.label[j]} {df.name[j][:14]}']
    print(title); show_grid(paths, titles, ncols=8, size=1.8)

cos_all, _ = cosine_scan(X_emb, thr=0.90)
for lo, hi in [(0.99, 1.01), (0.97, 0.99), (0.95, 0.97), (0.92, 0.95), (0.90, 0.92)]:
    show_pairs(cos_all[(cos_all[:, 2] >= lo) & (cos_all[:, 2] < hi)], title=f'cosine in [{lo}, {hi})')
for lo, hi in [(0, 3), (4, 6), (7, 10), (11, 14)]:
    show_pairs(pairs_any[(pairs_any[:, 2] >= lo) & (pairs_any[:, 2] <= hi)], title=f'pHash distance {lo}-{hi}')

**Выберите пороги по картинкам выше.** Правило: берём порог, при котором пары ещё
явно «то же фото» (тот же овощ, тот же кадр — просто сдвиг/кроп/яркость).
Лучше чуть перестраховаться: лишнее объединение делает оценку только строже.

In [ ]:
HAM_THR = 6      # <- adjust after looking at the pairs
COS_THR = 0.95   # <- adjust after looking at the pairs

ham_pairs = close_pairs_hamming(H, thr=HAM_THR)
cos_pairs = cos_all[cos_all[:, 2] >= COS_THR]
md5_pairs = np.array([(i, j) for idx in df.groupby('md5').indices.values() if len(idx) > 1
                      for i, j in zip(idx[:-1], idx[1:])]).reshape(-1, 2)

edges = np.concatenate([md5_pairs, ham_pairs[:, :2], cos_pairs[:, :2]]).astype(int)
n = len(df)
G = coo_matrix((np.ones(len(edges)), (edges[:, 0], edges[:, 1])), shape=(n, n))
n_groups, df['group'] = connected_components(G, directed=False)

gsize = df.group.value_counts()
print(f'pairs: md5={len(md5_pairs)}, pHash={len(ham_pairs)}, cosine={len(cos_pairs)}')
print(f'{n} images -> {n_groups} groups of near-duplicates ({n_groups / n:.1%} "unique" photos)')
print('group size distribution:', gsize.value_counts().sort_index().head(15).to_dict())
print('largest groups:', gsize.head(5).to_dict())
mixed = df.groupby('group').label.nunique()
print('groups that mix classes (label noise?):', (mixed > 1).sum())
uniq_per_class = df.groupby('label').group.nunique().reindex(CLASSES)
display(pd.DataFrame({'images': counts, 'unique_groups': uniq_per_class,
                      'images_per_group': (counts / uniq_per_class).round(2)}))
RESULTS['dedup'] = {'HAM_THR': HAM_THR, 'COS_THR': COS_THR, 'images': int(n), 'groups': int(n_groups),
                    'largest_group': int(gsize.iloc[0]), 'mixed_label_groups': int((mixed > 1).sum())}

⚠️ Если одна группа огромная (сотни фото) — порог слишком мягкий, «склеились» просто похожие фото
на одинаковом фоне. Поднимите `COS_THR` и перезапустите ячейку.

Посмотрим на самые большие группы — это обычно и есть «размноженные» фото.

In [ ]:
for g in gsize.index[:3]:
    members = df[df.group == g]
    show_grid(members.path.tolist()[:12], (members.label + ' ' + members.name.str[:14]).tolist()[:12], size=1.8)

### 1.5 Можно ли доверять готовому разбиению `final_dataset/` train/test?

Проверяем, есть ли в test фото, которые (почти) совпадают с фото в train. Если да — это **утечка**:
модель на тесте «узнаёт» картинки, а не распознаёт овощи.

In [ ]:
def light(path):
    try:
        raw = Path(path).read_bytes()
        return hashlib.md5(raw).hexdigest(), str(imagehash.phash(load_rgb(io.BytesIO(raw))))
    except Exception:
        return None, None

for d in (tr_df, te_df):
    r = Parallel(n_jobs=-1, batch_size=32)(delayed(light)(p) for p in tqdm(d.path))
    d['md5'], d['phash'] = zip(*r)
tr_ok, te_ok = tr_df.dropna().reset_index(drop=True), te_df.dropna().reset_index(drop=True)

exact = te_ok.md5.isin(set(tr_ok.md5))
near = close_pairs_hamming(hash_bits(te_ok.phash), hash_bits(tr_ok.phash), thr=HAM_THR)
near_idx = np.unique(near[:, 0]).astype(int)
print(f'test images: {len(te_ok)}')
print(f'  exact copy of a train image (md5): {exact.sum()} ({exact.mean():.1%})')
print(f'  near-copy of a train image (pHash <= {HAM_THR}): {len(near_idx)} ({len(near_idx) / len(te_ok):.1%})')

# If final_dataset files are byte copies of Dataset/, reuse our groups (also catches crops via embeddings)
g_of = dict(zip(df.md5, df.group))
tr_g, te_g = tr_ok.md5.map(g_of), te_ok.md5.map(g_of)
print(f'  final_dataset files found in Dataset/: train {tr_g.notna().mean():.0%}, test {te_g.notna().mean():.0%}')
leak_g = te_g.isin(set(tr_g.dropna()))
print(f'  test image whose near-duplicate GROUP is in train: {leak_g.sum()} ({leak_g.mean():.1%})')
RESULTS['author_split_leak'] = {'test': len(te_ok), 'exact': int(exact.sum()), 'near_phash': int(len(near_idx)),
                                'group_leak': int(leak_g.sum())}

### 1.6 Наше разбиение

Используем **разбиение по группам** (`StratifiedGroupKFold`): все копии одного фото попадают
только в одну часть — либо train, либо test. «Stratified» — пропорции классов одинаковые в частях.

**Для отчёта (п. 4: что помешает в магазине)** — посмотрите на сетку примеров выше и опишите своими словами:
фон (однотонный/студийный vs полка с ценниками), один овощ в кадре vs много, освещение, упаковка/плёнка,
ракурс и масштаб, класс «овоща нет» отсутствует вообще, EXIF-поворот у фото с телефона.

# Трек B. Классические модели поверх признаков

## B.1 + B.2. Два вида признаков × два вида валидации

- **Случайное разбиение** (`StratifiedKFold`): копии одного фото могут оказаться и в train, и в test.
- **Разбиение по группам** (`StratifiedGroupKFold`): копии всегда вместе.

Разница в цифрах ≈ насколько «случайная» оценка завышена из-за утечки.
Модель — `LogisticRegression` с `StandardScaler`, `class_weight='balanced'` (учёт дисбаланса, п. B.3).

In [ ]:
y = df.label.map({c: i for i, c in enumerate(CLASSES)}).values
groups = df.group.values

def make_model(C=1.0, balanced=True):
    return make_pipeline(StandardScaler(),
                         LogisticRegression(C=C, max_iter=3000, class_weight='balanced' if balanced else None))

def run_cv(X, splitter, use_groups):
    acc, f1 = [], []
    for tr, te in splitter.split(X, y, groups if use_groups else None):
        p = make_model().fit(X[tr], y[tr]).predict(X[te])
        acc.append(accuracy_score(y[te], p)); f1.append(f1_score(y[te], p, average='macro'))
    return np.mean(acc), np.std(acc), np.mean(f1), np.std(f1)

rows = []
for fname, X in [('simple: HSV hist + HOG', X_simple), ('ResNet18 embeddings', X_emb)]:
    for sname, spl, ug in [('random (StratifiedKFold)', StratifiedKFold(5, shuffle=True, random_state=SEED), False),
                           ('group (StratifiedGroupKFold)', StratifiedGroupKFold(5, shuffle=True, random_state=SEED), True)]:
        a, a_sd, f, f_sd = run_cv(X, spl, ug)
        rows.append({'features': fname, 'validation': sname, 'accuracy': f'{a:.3f} ± {a_sd:.3f}',
                     'macro_F1': f'{f:.3f} ± {f_sd:.3f}', '_f1': f})
        print(rows[-1])
res = pd.DataFrame(rows)
display(res.drop(columns='_f1'))
RESULTS['cv'] = res.drop(columns='_f1').to_dict('records')

## Отложенная выборка (holdout) для подробных метрик

Берём один фолд группового разбиения: ~80% групп — train, ~20% — test.

In [ ]:
tr_idx, te_idx = next(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(X_emb, y, groups))
assert not set(groups[tr_idx]) & set(groups[te_idx]), 'group leak!'
print('train', len(tr_idx), '| test', len(te_idx))

base = make_model().fit(X_emb[tr_idx], y[tr_idx])
pred = base.predict(X_emb[te_idx])
print(classification_report(y[te_idx], pred, target_names=CLASSES, digits=3))
ConfusionMatrixDisplay(confusion_matrix(y[te_idx], pred), display_labels=CLASSES).plot(xticks_rotation=45, cmap='Blues')
plt.title('Confusion matrix — embeddings + LogReg, group holdout'); plt.tight_layout()
plt.savefig(FIG / 'confusion_matrix.png', dpi=120); plt.show()

# Class imbalance: balanced vs not balanced
p_nb = make_model(balanced=False).fit(X_emb[tr_idx], y[tr_idx]).predict(X_emb[te_idx])
print('macro-F1 balanced    :', round(f1_score(y[te_idx], pred, average='macro'), 4))
print('macro-F1 not balanced:', round(f1_score(y[te_idx], p_nb, average='macro'), 4))
RESULTS['holdout'] = {'accuracy': accuracy_score(y[te_idx], pred), 'macro_f1': f1_score(y[te_idx], pred, average='macro'),
                      'report': classification_report(y[te_idx], pred, target_names=CLASSES, output_dict=True)}

In [ ]:
# Errors: what does the model confuse?
wrong = te_idx[pred != y[te_idx]]
if len(wrong):
    pred_of = dict(zip(te_idx, pred))
    w = wrong[:18]
    show_grid(df.path[w].tolist(), [f'true {CLASSES[y[i]]}\npred {CLASSES[pred_of[i]]}' for i in w],
              save='errors.png')

## B.3. Калибровка вероятностей

**Калибровка** — насколько можно верить `confidence`. Если модель говорит «90%» на 100 фото,
то правильно должно быть ~90 из них. **Reliability diagram**: по оси X — уверенность модели,
по оси Y — реальная доля правильных ответов. Идеально — диагональ.
**ECE** (expected calibration error) — среднее расстояние до диагонали (меньше = лучше).

Калибруем `CalibratedClassifierCV(method='sigmoid')` — фолды внутри train тоже по группам.

In [ ]:
def reliability(P, y_true, n_bins=10):
    conf, correct = P.max(1), P.argmax(1) == y_true
    ids = np.clip((conf * n_bins).astype(int), 0, n_bins - 1)
    xs, ys, ece = [], [], 0.0
    for b in range(n_bins):
        m = ids == b
        if m.any():
            xs.append(conf[m].mean()); ys.append(correct[m].mean())
            ece += m.mean() * abs(conf[m].mean() - correct[m].mean())
    return np.array(xs), np.array(ys), ece

inner = list(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(X_emb[tr_idx], y[tr_idx], groups[tr_idx]))
cal = CalibratedClassifierCV(make_model(), method='sigmoid', cv=inner).fit(X_emb[tr_idx], y[tr_idx])
P_raw, P_cal = base.predict_proba(X_emb[te_idx]), cal.predict_proba(X_emb[te_idx])

plt.figure(figsize=(5, 5)); plt.plot([0, 1], [0, 1], 'k--', lw=1, label='perfect')
ece = {}
for name, P in [('raw LogReg', P_raw), ('calibrated (sigmoid)', P_cal)]:
    xs, ys, e = reliability(P, y[te_idx]); ece[name] = e
    plt.plot(xs, ys, 'o-', label=f'{name}: ECE={e:.3f}, logloss={log_loss(y[te_idx], P, labels=range(6)):.3f}')
plt.xlabel('confidence'); plt.ylabel('accuracy'); plt.legend(fontsize=8); plt.title('Reliability diagram (group holdout)')
plt.tight_layout(); plt.savefig(FIG / 'reliability.png', dpi=120); plt.show()

USE_CAL = ece['calibrated (sigmoid)'] < ece['raw LogReg']
print('ECE:', {k: round(v, 4) for k, v in ece.items()}, '-> use calibration:', USE_CAL)
RESULTS['calibration'] = {**{k: float(v) for k, v in ece.items()}, 'use_calibration': bool(USE_CAL)}

## B.4. Кривая обучения: сколько фото на класс нужно?

Важно: берём **по одному фото из группы** (только уникальные фото). Иначе «100 фото» могли бы
оказаться 20 настоящими фото + 80 копий, и вывод «сколько снимать» был бы неверным.
Тест — та же групповая отложенная выборка.

In [ ]:
tr_unique = df.iloc[tr_idx].drop_duplicates('group').index.values     # one photo per group
avail = pd.Series(y[tr_unique]).value_counts().sort_index()
print('unique train photos per class:', dict(zip(CLASSES, avail.values)))

sizes = [s for s in [5, 10, 20, 50, 100, 200, 400, 800, 1600] if s <= avail.max()]
rng = np.random.default_rng(SEED)
lc = []
for n_pc in sizes:
    for rep in range(3):
        idx = np.concatenate([rng.permutation(tr_unique[y[tr_unique] == k])[:n_pc] for k in range(len(CLASSES))])
        p = make_model().fit(X_emb[idx], y[idx]).predict(X_emb[te_idx])
        lc.append({'n_per_class': n_pc, 'rep': rep, 'macro_f1': f1_score(y[te_idx], p, average='macro')})
lc = pd.DataFrame(lc).groupby('n_per_class').macro_f1.agg(['mean', 'std'])
display(lc.round(4))
plt.errorbar(lc.index, lc['mean'], yerr=lc['std'], marker='o', capsize=3)
plt.xscale('log'); plt.xlabel('unique photos per class (train)'); plt.ylabel('macro-F1 on group holdout')
plt.title('Learning curve — embeddings + LogReg'); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(FIG / 'learning_curve.png', dpi=120); plt.show()
RESULTS['learning_curve'] = lc.round(4).reset_index().to_dict('records')

## Бонус: ошибка «овощ есть, а его нет» в 5 раз дороже ложной тревоги

Пусть `p` — (откалиброванная!) вероятность, что овощ есть. Говорим «есть», если ожидаемый ущерб от этого меньше:

- сказали «есть», а его нет: ущерб `5 · (1 − p)`
- сказали «нет», а он есть: ущерб `1 · p`

«Есть» выгоднее, когда `5(1 − p) < p` → **`p > 5/6 ≈ 0.83`**. Без штрафов порог был бы 0.5.
Поэтому калибровка важна: порог 0.83 имеет смысл, только если «0.83» действительно означает 83%.

Ограничение: в датасете нет фото «овоща нет», поэтому проверить порог на реальных отказах нельзя —
ниже только видно, сколько правильных ответов мы «потеряем» на известных классах.
В `predict.py` порог включается флагом `--threshold 0.83`.

In [ ]:
P = P_cal if USE_CAL else P_raw
conf, ok = P.max(1), P.argmax(1) == y[te_idx]
ts = np.linspace(0.3, 0.99, 50)
kept = [(conf >= t).mean() for t in ts]
acc_kept = [ok[conf >= t].mean() if (conf >= t).any() else np.nan for t in ts]
plt.plot(ts, kept, label='share answered (not unknown)'); plt.plot(ts, acc_kept, label='accuracy on answered')
plt.axvline(5 / 6, ls='--', c='gray', label='5/6'); plt.xlabel('threshold'); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(FIG / 'threshold.png', dpi=120); plt.show()
t = 5 / 6
print(f'at threshold {t:.2f}: answered {(conf >= t).mean():.1%}, accuracy on answered {ok[conf >= t].mean():.3f}')

# Финальная модель, predict.py, замер на CPU

Финальную модель учим на **всех** данных (метрики выше уже честно посчитаны на групповом разбиении).

In [ ]:
import sklearn, PIL, joblib
cv_all = list(StratifiedGroupKFold(5, shuffle=True, random_state=SEED).split(X_emb, y, groups))
final = (CalibratedClassifierCV(make_model(), method='sigmoid', cv=cv_all) if USE_CAL else make_model()).fit(X_emb, y)
joblib.dump({'model': final, 'classes': CLASSES}, REPO / 'weights' / 'classifier.joblib')
torch.save({k: v.cpu() for k, v in backbone.state_dict().items()}, REPO / 'weights' / 'backbone_resnet18.pt')

v = lambda m: m.__version__.split('+')[0]
(REPO / 'requirements.txt').write_text(
    '--extra-index-url https://download.pytorch.org/whl/cpu\n'
    f'torch=={v(torch)}\ntorchvision=={v(torchvision)}\nscikit-learn=={v(sklearn)}\n'
    f'numpy=={v(np)}\npillow=={v(PIL)}\njoblib=={v(joblib)}\n')
import scipy, skimage, matplotlib
(REPO / 'requirements-train.txt').write_text(
    '-r requirements.txt\n'
    f'pandas=={v(pd)}\nscipy=={v(scipy)}\nscikit-image=={v(skimage)}\nmatplotlib=={v(matplotlib)}\n'
    f'imagehash=={v(imagehash)}\nkagglehub\ntqdm\n')
print((REPO / 'requirements.txt').read_text())
for f in sorted((REPO / 'weights').iterdir()): print(f.name, round(f.stat().st_size / 1e6, 1), 'MB')

In [ ]:
%%writefile /content/repo/predict.py
"""Vegetable classifier: frozen ResNet18 embeddings + (calibrated) logistic regression.

Usage:
    python predict.py --input path/to/folder --output preds.csv [--threshold 0.83]

Runs on CPU, no internet needed: all weights are loaded from ./weights.
"""
import argparse
import csv
import time
from pathlib import Path

import joblib
import numpy as np
import torch
from PIL import Image, ImageOps
from torchvision import models, transforms

HERE = Path(__file__).resolve().parent
EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

# Same preprocessing as in training (see notebook)
TF = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


def load_rgb(path, max_side=320):
    img = Image.open(path)
    img.draft("RGB", (max_side, max_side))  # fast JPEG downscale while decoding
    img = ImageOps.exif_transpose(img)      # phone photos: respect EXIF rotation
    return img.convert("RGB")


def load_backbone(weights_path):
    net = models.resnet18(weights=None)
    net.fc = torch.nn.Identity()
    net.load_state_dict(torch.load(weights_path, map_location="cpu"))
    return net.eval()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--input", required=True, help="folder with images")
    ap.add_argument("--output", required=True, help="path to preds.csv")
    ap.add_argument("--weights-dir", default=str(HERE / "weights"))
    ap.add_argument("--threshold", type=float, default=0.0,
                    help="confidence below this -> 'unknown' (0 = never unknown)")
    ap.add_argument("--batch-size", type=int, default=32)
    args = ap.parse_args()

    t_start = time.perf_counter()
    wdir = Path(args.weights_dir)
    net = load_backbone(wdir / "backbone_resnet18.pt")
    bundle = joblib.load(wdir / "classifier.joblib")
    clf, classes = bundle["model"], bundle["classes"]
    t_loaded = time.perf_counter()

    root = Path(args.input)
    files = sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in EXTS)
    rows = []

    for s in range(0, len(files), args.batch_size):
        batch = files[s:s + args.batch_size]
        tensors, names = [], []
        for p in batch:
            name = p.relative_to(root).as_posix()
            try:
                tensors.append(TF(load_rgb(p)))
                names.append(name)
            except Exception as e:  # broken file: do not crash the whole run
                print(f"WARNING: cannot read {name}: {e}")
                rows.append((name, "unknown", 0.0))
        if not tensors:
            continue
        with torch.inference_mode():
            emb = net(torch.stack(tensors)).numpy()
        probs = clf.predict_proba(emb)
        for name, pr in zip(names, probs):
            k = int(np.argmax(pr))
            conf = float(pr[k])
            label = classes[k] if conf >= args.threshold else "unknown"
            rows.append((name, label, round(conf, 4)))

    rows.sort(key=lambda r: r[0])
    with open(args.output, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["filename", "label", "confidence"])
        w.writerows(rows)

    t_end = time.perf_counter()
    n = max(len(files), 1)
    print(f"images: {len(files)} -> {args.output}")
    print(f"model load: {t_loaded - t_start:.2f} s")
    print(f"per image (read + preprocess + predict, CPU): {1000 * (t_end - t_loaded) / n:.1f} ms")


if __name__ == "__main__":
    main()

In [ ]:
# CPU timing + sanity check on 60 holdout photos
sample_dir = Path('/content/sample_cpu'); shutil.rmtree(sample_dir, ignore_errors=True); sample_dir.mkdir()
pick = np.random.default_rng(SEED).choice(te_idx, 60, replace=False)
for i in pick: shutil.copy(df.path[i], sample_dir / f'{i}{Path(df.path[i]).suffix}')
!CUDA_VISIBLE_DEVICES="" python /content/repo/predict.py --input /content/sample_cpu --output /content/preds_sample.csv
preds = pd.read_csv('/content/preds_sample.csv')
preds['true'] = preds.filename.map(lambda f: df.label[int(Path(f).stem)])
print('accuracy on sample:', (preds.label == preds.true).mean()); display(preds.head())

In [ ]:
(REPO / 'results.json').write_text(json.dumps(RESULTS, indent=2, default=float))
shutil.make_archive('/content/repo', 'zip', REPO)
from google.colab import files; files.download('/content/repo.zip')

**Не забудьте:** File → Download → .ipynb, положите ноутбук в репозиторий рядом с `predict.py`.